# 01 SFT Ablation

Purpose: run the FinChain SFT baseline and checkpoint-selection flow for Qwen2.5-1.5B-style small-model experiments.

Expected inputs: `configs/finchain/sft/qwen25_1_5b_sft.yaml` and `data/finchain_v2_template_disjoint/`.

Expected outputs: candidate checkpoints, validation/test eval summaries, and a selected checkpoint intended for `shannan-liu1/qwen25-1p5b-finchain-v2-sft-selected`.

HF status: selected and candidate/eval repos are listed in `docs/hf_checkpoints.md`.

## Before you run a single cell in this notebook - terminal pre-flight

This notebook runs FinChain Supervised Fine-Tuning (SFT) on `Qwen/Qwen2.5-1.5B`, producing the HF-format checkpoint used by DPO, OPD/GKD, and GRPO. Run this terminal block first on a fresh GPU environment. It pins the CUDA/PyTorch stack before NCCL work, keeps HF cache on the persistent volume, requires the FinChain v2 template-disjoint JSONLs and manifest under the local data path, and authenticates W&B/Hugging Face before the trainer starts. If you are restarting a stopped environment or recovering from a package failure, rerun this block before resuming training; do not jump straight to an Accelerate launch.

```bash
cd /workspace
test -d finpost || git clone https://github.com/shannan-liu1/finpost.git
cd /workspace/finpost
git checkout main
git pull --ff-only

# Install project deps, then fail fast on CUDA/NCCL drift. If the guard fails,
# the repair script removes CUDA 13 pip packages and reinstalls the A40-safe
# Torch CUDA 12.4 stack:
#   torch==2.6.0+cu124
# The repair script also removes optional torchvision/torchaudio wheels;
# finpost does not use them, and broken optional wheels can make transformers imports fail.
python -m pip install -e ".[dev,rlvr,chaineval]"
nvidia-smi || true
bash scripts/repair_cuda_stack.sh
python -m pip install -e ".[dev,rlvr,chaineval]"
python scripts/check_cuda_stack.py

# If nvidia-smi is missing or shows no GPUs, stop and fix the environment/image. The
# repair script fixes Python wheel/runtime drift; it cannot install the host GPU driver.

# Persistent cache + FinChain split paths. The public repo does not
# track these JSONLs; generate or place them under data/finchain_v2_template_disjoint
# before the paid GPU run.
export HF_HOME=/workspace/hf-cache
export WANDB_MODE=${WANDB_MODE:-online}
export WANDB_PROJECT=finpost-finchain-sft
mkdir -p /workspace/data/finchain_v2_template_disjoint /workspace/hf-cache
for f in train.jsonl validation.jsonl test.jsonl manifest.json; do
  test -f "data/finchain_v2_template_disjoint/$f" || { echo "Missing data/finchain_v2_template_disjoint/$f; generate or place the FinChain v2 splits before training." >&2; exit 1; }
  cp -n "data/finchain_v2_template_disjoint/$f" /workspace/data/finchain_v2_template_disjoint/
done
export FINPOST_FINCHAIN_TRAIN_JSONL=/workspace/data/finchain_v2_template_disjoint/train.jsonl
export FINPOST_FINCHAIN_VALIDATION_JSONL=/workspace/data/finchain_v2_template_disjoint/validation.jsonl
export FINPOST_FINCHAIN_TEST_JSONL=/workspace/data/finchain_v2_template_disjoint/test.jsonl
python scripts/audit_finchain_template_disjoint_manifest.py --data-dir data/finchain_v2_template_disjoint --out artifacts/preflight/finchain_v2_manifest_audit.json
python scripts/gpu_preflight.py --out artifacts/preflight/preflight_report.json --timeout-sec 900

# Auth. wandb status should show your username. huggingface-cli whoami should
# succeed if you plan to push checkpoints or access private/gated repos.
wandb login
wandb status
huggingface-cli login
huggingface-cli whoami

# Pre-download model snapshots so an Accelerate launch does not look hung while
# it is only fetching weights.
HF_HOME=/workspace/hf-cache python scripts/warm_hf_cache.py Qwen/Qwen2.5-0.5B Qwen/Qwen2.5-1.5B
```

Hardware target: 2x A40 48 GB for the distributed TRL path below. Use `NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1` as the safe 2x A40 starting point; remove those only after a small distributed canary passes without the NCCL/CUDA-driver error. Do **not** pass `--packing` unless `flash-attn` is installed and a packed canary has already passed.


# FinChain SFT GPU Notebook

Runs SFT on the FinChain v2 template-disjoint split, producing `results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu/selected`. That HF-format checkpoint is the policy/reference starting point for the DPO, OPD/GKD, and GRPO notebooks.

Recommended paid-GPU path: terminal preflight -> sanity check -> CUDA/data/auth guard -> HF cache warmup -> module/data check -> 0.5B stack canary -> 1.5B distributed canary -> full distributed SFT -> checkpoint smoke-load -> eval vs base -> optional Hub push -> stop instance.


## Step 1 - Sanity-check the environment

In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import json
import os
import platform
import subprocess
import sys
import time

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/workspace/finpost") if Path("/workspace/finpost").exists() else PROJECT_ROOT
os.chdir(PROJECT_ROOT)

RESULTS_DIR = PROJECT_ROOT / "results" / "finchain_sft"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def progress(title, detail=None):
    stamp = time.strftime("%H:%M:%S")
    print(f"[{stamp}] {title}")
    if detail:
        print(detail)


def run_cmd(cmd, *, check=False):
    progress("running command", cmd)
    completed = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with exit {completed.returncode}: {cmd}")
    return completed


def append_cost_event(stage, **payload):
    path = RESULTS_DIR / "cost_ledger.jsonl"
    row = {"stage": stage, "time": time.strftime("%Y-%m-%dT%H:%M:%S"), **payload}
    with path.open("a", encoding="utf-8") as fp:
        fp.write(json.dumps(row, sort_keys=True) + "\n")
    print(json.dumps(row, indent=2, sort_keys=True))
    return row


progress("project root", str(PROJECT_ROOT))
progress("python", sys.version.split()[0])
progress("platform", platform.platform())

## Distributed launch preflight

Run this after the setup cell and before any multi-GPU command. If it reports fewer than 2 CUDA devices, stay on the single-GPU path. Treat distributed training as verified only after an Accelerate/TRL or Accelerate/DDP multi-process command has produced checkpoints.


In [ ]:
progress("distributed launch preflight")
try:
    import accelerate
    print("accelerate:", accelerate.__version__)
except Exception as exc:
    print("accelerate import failed:", repr(exc))

try:
    import torch
    print("cuda devices:", torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(f"gpu {idx}: {props.name}, vram={props.total_memory / 1e9:.1f} GB")
except Exception as exc:
    print("torch cuda check failed:", repr(exc))

for key in ["CUDA_VISIBLE_DEVICES", "WORLD_SIZE", "LOCAL_RANK", "RANK"]:
    print(f"{key}={os.environ.get(key)}")

run_cmd("accelerate env")


## GPU setup guardrails

Run this before any full GPU run. It catches the failure modes that waste GPU time:

- CUDA/PyTorch first: run `python scripts/check_cuda_stack.py`; if it fails in the environment, run `bash scripts/repair_cuda_stack.sh` and re-check before `accelerate launch`.
- Correct CUDA wheel baseline for current CUDA 12.x environments: Torch CUDA 12.4 with `torch==2.6.0+cu124`. ChainEval/BERTScore needs torch >= 2.6, and broken optional `torchvision`/`torchaudio` wheels can make transformers imports fail. Avoid CUDA 13 pip packages next to a Torch CUDA 12 build.
- Hugging Face cache: keep `HF_HOME=/workspace/hf-cache` and pre-download models with `python scripts/warm_hf_cache.py ...` so download time is not confused with a training stall.
- Auth before launch: run `wandb login`, `wandb status`, `huggingface-cli login`, and `huggingface-cli whoami` in the terminal preflight. In-notebook commands below check status but do not block if you intentionally run offline.
- FinChain data: the public repo does not track the split JSONLs. Generate or place `train.jsonl`, `validation.jsonl`, `test.jsonl`, and `manifest.json` under `data/finchain_v2_template_disjoint/`; this notebook copies them to `/workspace/data/finchain_v2_template_disjoint` if the runtime data path is empty, then sets the loader env vars.
- Fast canaries: use `Qwen/Qwen2.5-0.5B` for cheap CUDA/data/W&B plumbing, then run the 1.5B distributed canary before the paid 750-step run.
- Distributed SFT wiring: `accelerate launch --num_processes 2 scripts/train_finchain_trl_sft.py` starts one process per GPU. TRL/Accelerate handle DDP optimizer synchronization, bf16 mixed precision, and HF-format checkpoint writes under `output_dir/final`.
- Eval parallelism defaults to example sharding in `scripts/run_finchain_eval_parallel.py --gpus 0 1 --parallelism examples`: it splits rows across GPUs for each checkpoint and merges artifacts. This is data sharding, not model sharding.


In [ ]:
import os
import shutil
from pathlib import Path

os.environ.setdefault("HF_HOME", "/workspace/hf-cache")
os.environ.setdefault("WANDB_MODE", "online")
os.environ.setdefault("WANDB_PROJECT", "finpost-finchain-sft")

runtime_data_dir = Path("/workspace/data/finchain_v2_template_disjoint")
repo_data_dir = PROJECT_ROOT / "data" / "finchain_v2_template_disjoint"
runtime_data_dir.mkdir(parents=True, exist_ok=True)

missing_inputs = []
for split in ["train", "validation", "test"]:
    source = repo_data_dir / f"{split}.jsonl"
    target = runtime_data_dir / f"{split}.jsonl"
    if not target.exists() and source.exists():
        shutil.copy2(source, target)
    os.environ[f"FINPOST_FINCHAIN_{split.upper()}_JSONL"] = str(target)
    if not target.exists():
        missing_inputs.append(str(target))

manifest_source = repo_data_dir / "manifest.json"
manifest_target = runtime_data_dir / "manifest.json"
if not manifest_target.exists() and manifest_source.exists():
    shutil.copy2(manifest_source, manifest_target)
if not manifest_target.exists():
    missing_inputs.append(str(manifest_target))

if missing_inputs:
    raise FileNotFoundError(
        "Missing FinChain v2 split JSONLs or manifest. Generate or place them under "
        "data/finchain_v2_template_disjoint/ before training: "
        + ", ".join(missing_inputs)
    )

progress("GPU CUDA/data guard")
run_cmd("python scripts/check_cuda_stack.py", check=True)

progress("auth status")
run_cmd("wandb status", check=False)
run_cmd("huggingface-cli whoami", check=False)

for key in [
    "HF_HOME",
    "WANDB_MODE",
    "WANDB_PROJECT",
    "FINPOST_FINCHAIN_TRAIN_JSONL",
    "FINPOST_FINCHAIN_VALIDATION_JSONL",
    "FINPOST_FINCHAIN_TEST_JSONL",
]:
    print(f"{key}={os.environ.get(key)}")


## Step 1.5 - Hugging Face cache warmup

Run once per environment/volume. This downloads tokenizer/config/safetensors into `HF_HOME` without loading the model on GPU. After this cell, a long pause during `accelerate launch` is trainer setup or generation, not model download.


In [ ]:
HF_WARMUP_MODELS = ["Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-1.5B"]
progress("HF cache warmup", " ".join(HF_WARMUP_MODELS))
warm_cmd = "HF_HOME=/workspace/hf-cache python scripts/warm_hf_cache.py " + " ".join(HF_WARMUP_MODELS)
run_cmd(warm_cmd, check=True)


In [ ]:
progress("GPU preflight")
try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            print(f"gpu {idx}: {props.name}, vram={props.total_memory / 1e9:.1f} GB")
except Exception as exc:
    print("torch check failed:", repr(exc))

run_cmd("nvidia-smi")

In [ ]:
# Confirm core modules + FinChain data are accessible. If any import or the
# load_finchain call fails, fix it before any GPU work below.
for module_name in [
    "finpost.training.train",
    "finpost.training.trainer",
    "finpost.training.dataset",
    "finpost.data.finchain_dataset",
    "finpost.evals.finchain_metrics",
]:
    try:
        importlib.import_module(module_name)
        print(module_name, "OK")
    except Exception as exc:
        print(module_name, "FAILED:", repr(exc))

from finpost.data.finchain_dataset import load_finchain, resolve_finchain_path

for split in ["train", "validation", "test"]:
    path = resolve_finchain_path(split)
    print(split, "->", path, "exists=", path.exists())
split_examples = {split: load_finchain(split) for split in ["train", "validation", "test"]}
assert len(split_examples["train"]) == 2320, "expected FinChain v2 train split to contain 2320 rows"
assert len(split_examples["validation"]) == 870, "expected FinChain v2 validation split to contain 870 rows"
assert len(split_examples["test"]) == 870, "expected FinChain v2 test split to contain 870 rows"
train_examples = split_examples["train"]
print("train examples:", len(train_examples))
print("first prompt id:", train_examples[0].id)
print("first gold answer:", train_examples[0].final_answer)

## Step 2 - Hyperparameters

Defaults come from `configs/finchain/sft/qwen25_1_5b_sft.yaml`. The YAML is the reference for the full run; this cell exists so you can override paths if your environment uses different result directories.

In [ ]:
SFT_CONFIG = {
    "base_model_id": "Qwen/Qwen2.5-1.5B",
    "fast_canary_model_id": "Qwen/Qwen2.5-0.5B",
    "sft_config_yaml": "configs/finchain/sft/qwen25_1_5b_sft.yaml",
    "save_dir": "results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu",
    "hf_dir": "results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu/selected",
    # The YAML already pins these, listed here for reference:
    "max_steps": 750,
    "warmup_steps": 75,
    "lr": 2.0e-5,
    "per_device_batch_size": 1,
    "grad_accum_steps": 8,
    "max_seq_len": 1536,
    "wandb_project": "finpost-finchain-sft",
    "run_name": "qwen25-1p5b-finchain-v2-trl-sft-2gpu",
    "distributed_run_name": "qwen25-1p5b-finchain-v2-trl-sft-2gpu",
}
print(json.dumps(SFT_CONFIG, indent=2))


## Step 3 - Fast distributed SFT stack canary (0.5B, 5 steps)

Catches CUDA/NCCL, FinChain data, tokenizer cache, TRL SFT config, and W&B-offline plumbing without loading the 1.5B model. This is a cheap stack canary, not part of the reported results.


In [ ]:
fast_canary_out = "results/checkpoints/qwen25-0p5b-finchain-trl-sft-canary"
fast_canary_cmd = """
WANDB_MODE=offline HF_HOME=/workspace/hf-cache NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 accelerate launch   --num_processes 2   --mixed_precision bf16   scripts/train_finchain_trl_sft.py   --model Qwen/Qwen2.5-0.5B   --train-n 64   --output-dir results/checkpoints/qwen25-0p5b-finchain-trl-sft-canary   --max-steps 5   --per-device-train-batch-size 1   --gradient-accumulation-steps 2   --max-length 512   --save-steps 999   --logging-steps 1   --report-to none   --run-name qwen25-0p5b-finchain-trl-sft-canary-2xa40
"""
fast_start = time.perf_counter()
fast_result = run_cmd(fast_canary_cmd, check=False)
append_cost_event(
    "sft_fast_distributed_canary_0p5b",
    elapsed_sec=round(time.perf_counter() - fast_start, 1),
    exit_code=fast_result.returncode,
    out_dir=fast_canary_out,
)
if fast_result.returncode != 0:
    raise RuntimeError("Fast distributed SFT canary failed. Do not run the 1.5B canary/full training.")


## Step 3.5 - Real model distributed SFT canary (1.5B, 5 steps)

Runs the same distributed stack with `Qwen/Qwen2.5-1.5B`, tiny data, short sequence length, no W&B, and no checkpoint save. If this fails or hangs before W&B/logging, fix CUDA/NCCL/HF cache before the paid full run.


In [ ]:
real_canary_out = "results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-canary"
real_canary_cmd = """
WANDB_MODE=offline HF_HOME=/workspace/hf-cache NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 accelerate launch   --num_processes 2   --mixed_precision bf16   scripts/train_finchain_trl_sft.py   --model Qwen/Qwen2.5-1.5B   --train-n 64   --output-dir results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-canary   --max-steps 5   --per-device-train-batch-size 1   --gradient-accumulation-steps 2   --max-length 1024   --save-steps 999   --logging-steps 1   --report-to none   --run-name qwen25-1p5b-finchain-v2-trl-sft-canary-2xa40
"""
real_start = time.perf_counter()
real_result = run_cmd(real_canary_cmd, check=False)
append_cost_event(
    "sft_real_distributed_canary_1p5b",
    elapsed_sec=round(time.perf_counter() - real_start, 1),
    exit_code=real_result.returncode,
    out_dir=real_canary_out,
)
if real_result.returncode != 0:
    raise RuntimeError("1.5B distributed SFT canary failed. Do not run full training.")


## Step 4 - Full distributed SFT training run

This is the main SFT artifact path: 2 GPU processes via Accelerate/TRL, bf16, no packing by default, W&B online, checkpoint every 250 steps, final HF-format checkpoint under `results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu/selected`.

Global batch = `per_device_train_batch_size * gradient_accumulation_steps * num_processes` = `1 * 8 * 2 = 16` examples per optimizer step. Do not add `--packing` unless `flash-attn` is installed and a packed canary has already passed.


## Step 4.5 - Optional repo-native SFT path

The repo-native trainer is the readable first-principles implementation, but it is single-process. Keep it as a debugging/reference path. Do **not** run it after the distributed TRL full run unless you intentionally want a second separate training job.


In [ ]:
# Full two-GPU distributed SFT. This is the default paid run path.
# Safe A40 defaults: HF cache on volume, NCCL P2P/IB disabled, no TRL packing.
import os

os.environ["WANDB_MODE"] = "online"
os.environ["WANDB_PROJECT"] = SFT_CONFIG["wandb_project"]

sft_dist_cmd = f"""
HF_HOME=/workspace/hf-cache NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 accelerate launch   --num_processes 2   --mixed_precision bf16   scripts/train_finchain_trl_sft.py   --model {SFT_CONFIG['base_model_id']}   --train-n 2320   --output-dir results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu   --max-steps 750   --per-device-train-batch-size 1   --gradient-accumulation-steps 8   --max-length 1536   --save-steps 250   --logging-steps 10   --report-to wandb   --run-name {SFT_CONFIG['distributed_run_name']}
"""

dist_start = time.perf_counter()
dist_result = run_cmd(sft_dist_cmd, check=True)
append_cost_event(
    "sft_full_distributed_run",
    elapsed_sec=round(time.perf_counter() - dist_start, 1),
    exit_code=dist_result.returncode,
)


In [ ]:
# repo_native_cmd = (
#     "python -m finpost.training.train "
#     f"--config {SFT_CONFIG['sft_config_yaml']} --device cuda"
# )
# print("Repo-native single-process SFT command is staged for debugging only:")
# print(repo_native_cmd)
# print("Do not run this after the distributed TRL full run unless you want a second separate job.")


## Step 5 - Verify checkpoints landed

Confirms the atomic-write pattern produced complete `step-NNNNNNNN/` directories containing `model.safetensors` and `state.pt`. Picks the latest step as the one to convert and evaluate.

In [ ]:
# TRL writes each candidate in Hugging Face format, so validation can score all of them.
trl_dir = Path("results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu")
hf_dir = trl_dir / "final"
candidate_dirs = {
    f"sft_{path.name.replace('-', '_')}": path
    for path in sorted(trl_dir.glob("checkpoint-*"))
}
candidate_dirs["sft_final"] = hf_dir

for label, path in candidate_dirs.items():
    model_files = list(path.glob("*.safetensors")) + list(path.glob("pytorch_model*.bin"))
    print(label, path, "weights=", bool(model_files))
    if not path.exists() or not model_files:
        raise RuntimeError(f"missing or incomplete SFT checkpoint: {path}")
print("candidate checkpoints:", list(candidate_dirs))


## Step 6 - Smoke-load the TRL HF checkpoint

The TRL distributed path writes Hugging Face-format candidate checkpoints, including `results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu/final`. Smoke-load the final candidate before validation selection so you know the artifact is usable.

In [ ]:
# Smoke-load the saved TRL HF checkpoint before eval or shutdown.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["HF_HOME"] = "/workspace/hf-cache"

tok = AutoTokenizer.from_pretrained(hf_dir, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    hf_dir,
    dtype=torch.bfloat16,
    local_files_only=True,
).to("cuda:0")

print("loaded checkpoint:", hf_dir)
print("tokenizer vocab:", len(tok))
print("vram GB:", round(torch.cuda.memory_allocated() / 1e9, 2))

del model
torch.cuda.empty_cache()


In [ ]:
# No conversion is needed for the TRL path. The smoke-load cell above is the verifier.
print("TRL checkpoint is already in Hugging Face format:", hf_dir)


## Optional - Restore an existing Hub SFT model for diagnostic evaluation

Leave this off for the v2 rerun. Enable it only when you want to
evaluate the previously saved SFT model on the new validation/test protocol;
that result is diagnostic because the model was trained on the earlier split.


In [ ]:
EVALUATE_EXISTING_HUB_MODEL = False
PILOT_SFT_REPO_ID = "shannan-liu1/qwen25-1p5b-finchain-sft"

if EVALUATE_EXISTING_HUB_MODEL:
    from huggingface_hub import snapshot_download

    restored_dir = Path("results/checkpoints/restored/qwen25-1p5b-finchain-sft-pilot")
    snapshot_download(repo_id=PILOT_SFT_REPO_ID, local_dir=str(restored_dir))
    candidate_dirs = {"sft_hub_pilot": restored_dir}
    print("Diagnostic only: restored pilot model was not trained on the v2 split.")


## Step 7 - Select on validation, then report the untouched test result

Validation is doing one concrete job: select among saved SFT checkpoints using
deterministic final-answer `accuracy` on all 870 validation rows. The selected
HF artifact is copied to `selected/` for DPO, OPD/GKD, and GRPO. Test is then
run once for the selected policy against base, with ChainEval enabled to report
step precision, recall, F1, ROUGE/BERTScore, and alignment metrics.


In [ ]:
import shutil
import torch as _t

EVAL_ROOT = PROJECT_ROOT / "results" / "evals" / "sft_v2_template_disjoint"
VALIDATION_OUT_DIR = EVAL_ROOT / "validation_selection"
TEST_OUT_DIR = EVAL_ROOT / "test_selected_chaineval"
EVAL_CONFIG = {"parallelism": "examples", "batch_size_finchain": 32}
gpus_arg = "0 1" if _t.cuda.is_available() and _t.cuda.device_count() > 1 else "0"

candidate_args = " ".join(f"{label}={path}" for label, path in candidate_dirs.items())
validation_cmd = (
    "HF_HOME=/workspace/hf-cache "
    "python scripts/run_finchain_eval_parallel.py "
    f"--gpus {gpus_arg} --parallelism {EVAL_CONFIG['parallelism']} "
    f"--checkpoints {candidate_args} --finchain-split validation --n 870 "
    f"--out-dir {VALIDATION_OUT_DIR} "
    f"--batch-size-finchain {EVAL_CONFIG['batch_size_finchain']}"
)
eval_start = time.perf_counter()
run_cmd(validation_cmd, check=True)
append_cost_event("sft_v2_validation_selection", elapsed_sec=round(time.perf_counter() - eval_start, 1))

validation_summaries = {
    label: json.loads((VALIDATION_OUT_DIR / label / "accuracy_summary.json").read_text(encoding="utf-8"))[0]
    for label in candidate_dirs
}

def checkpoint_step(label: str) -> int:
    return int(label.rsplit("_", 1)[-1]) if "checkpoint_" in label else 10**12

selected_label = sorted(
    validation_summaries,
    key=lambda label: (
        -validation_summaries[label]["accuracy"],
        -validation_summaries[label].get("parse_success_rate", 0.0),
        checkpoint_step(label),
    ),
)[0]
selected_dir = candidate_dirs[selected_label]
if not EVALUATE_EXISTING_HUB_MODEL:
    selected_hf_dir = trl_dir / "selected"
    if selected_hf_dir.exists():
        shutil.rmtree(selected_hf_dir)
    shutil.copytree(selected_dir, selected_hf_dir)
    selected_dir = selected_hf_dir
print("selected on validation:", selected_label, selected_dir)

selection_metadata = {
    "selection_split": "validation",
    "final_eval_split": "test",
    "selected_checkpoint": str(selected_dir),
    "selected_label": selected_label,
    "selection_metric": "accuracy",
    "tie_breaker": "parse_success_rate_then_earliest_step",
    "test_touched_before_selection": False,
}
(EVAL_ROOT / "selection_metadata.json").write_text(
    json.dumps(selection_metadata, indent=2, sort_keys=True),
    encoding="utf-8",
)

test_cmd = (
    "HF_HOME=/workspace/hf-cache "
    "python scripts/run_finchain_eval_parallel.py "
    f"--gpus {gpus_arg} --parallelism {EVAL_CONFIG['parallelism']} "
    f"--checkpoints base=Qwen/Qwen2.5-1.5B sft_selected={selected_dir} "
    "--finchain-split test --n 870 "
    f"--out-dir {TEST_OUT_DIR} "
    f"--batch-size-finchain {EVAL_CONFIG['batch_size_finchain']} --enable-chaineval"
)
eval_start = time.perf_counter()
run_cmd(test_cmd, check=True)
append_cost_event("sft_v2_test_selected_chaineval", elapsed_sec=round(time.perf_counter() - eval_start, 1))


## Step 8 - Headline numbers

Use `validation_summaries` only for checkpoint selection. The publishable
base-versus-SFT comparison is in `TEST_OUT_DIR`; `accuracy` is final-answer
correctness, while ChainEval fields describe reasoning-chain quality.


In [ ]:
HEADLINE_METRICS = ["n_evaluated", "accuracy", "parse_success_rate", "step_recall", "step_precision", "step_f1"]

def compact_metrics(summary):
    return {metric: summary.get(metric) for metric in HEADLINE_METRICS if metric in summary}

test_summaries = {}
for name in ["base", "sft_selected"]:
    summary_path = TEST_OUT_DIR / name / "accuracy_summary.json"
    test_summaries[name] = json.loads(summary_path.read_text(encoding="utf-8"))[0]
print("validation selection:")
print(json.dumps(validation_summaries, indent=2, sort_keys=True))
print("untouched test summary metrics:")
print(json.dumps({name: compact_metrics(summary) for name, summary in test_summaries.items()}, indent=2, sort_keys=True))
print("untouched test full ChainEval summaries:")
print(json.dumps(test_summaries, indent=2, sort_keys=True))


## Push the SFT checkpoint to HF Hub

This uploads the HF-format model folder to `shannan-liu1/qwen25-1p5b-finchain-v2-sft-selected`. Run only after the checkpoint smoke-load/eval cell succeeds and `huggingface-cli whoami` shows the account that can write to `shannan-liu1`.


In [ ]:
from huggingface_hub import HfApi

repo_id = "shannan-liu1/qwen25-1p5b-finchain-v2-sft-selected"
folder_to_push = Path(selected_dir)
if not folder_to_push.exists():
    raise FileNotFoundError(f"checkpoint folder does not exist: {folder_to_push}")

api = HfApi()
api.create_repo(repo_id, exist_ok=True, private=False)
api.upload_folder(
    folder_path=str(folder_to_push),
    repo_id=repo_id,
    repo_type="model",
    commit_message="FinChain v2 SFT validation-selected HF checkpoint",
)
print(f"pushed {folder_to_push} to https://huggingface.co/{repo_id}")
append_cost_event("hf_push_sft_v2_selected", repo_id=repo_id, folder=str(folder_to_push))


## Final - Stop the instance (or move to DPO / GRPO / GKD)

GPU rentals can charge by the hour. If you only needed SFT, stop the instance through your provider console or CLI. If you plan to continue with DPO (`notebooks/02_dpo.ipynb`), GRPO/RLVR (`notebooks/03_grpo_rlvr.ipynb`), or OPD/GKD (`notebooks/04_opd_gkd.ipynb`) on the same environment, keep it running; those notebooks use `results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu/selected`, which now exists locally after the distributed TRL run.